In [1]:
from utils.blablador import Models
from langchain.chat_models import ChatOpenAI
from config.config import API_KEY

models = Models(api_key=API_KEY).get_model_ids()
llm = ChatOpenAI(
    model_name=models[8],
    base_url="https://helmholtz-blablador.fz-juelich.de:8000/v1",
    api_key=API_KEY,
    #  flatten_messages_as_text=True,
    temperature=0.2)
response = llm.invoke("What is the capital of Taiwan?")
print(response)

KeyError: 'data'

In [18]:
from langchain.chat_models import ChatOpenAI
from typing import List, Optional
from config.config import API_KEY
import requests
import json


class BlabladorClient:
    """Client for interacting with the Blablador API."""

    def __init__(self, api_key: str):
        """Initialize the Blablador client.
        
        Args:
            api_key: Your Blablador API key
        """
        self.api_key = api_key
        self.api_base = "https://helmholtz-blablador.fz-juelich.de:8000/v1"
        self.headers = {
            'accept': 'application/json',
            'Authorization': f'Bearer {api_key}'
        }

    def list_models(self) -> List[str]:
        """Get available model IDs from Blablador.
        
        Returns:
            List of available model IDs
        """
        # This is a placeholder - in a real implementation you would
        # make an API call to fetch available models
        # For example:
        # import requests
        # response = requests.get(f"{self.api_base}/models",
        #                         headers={"Authorization": f"Bearer {self.api_key}"})
        # return [model["id"] for model in response.json()["data"]]

        # For now, returning common model IDs that might be available
        response = requests.get(url=self.api_base, headers=self.headers)
        response = json.loads(response.text)
        print(response)

        ids = []
        for model in response["data"]:
            ids.append(model["id"])

        return (ids)

    def get_chat_model(self,
                       model_id: Optional[str] = None,
                       temperature: float = 0.7) -> ChatOpenAI:
        """Create a configured ChatOpenAI instance for the specified model.
        
        Args:
            model_id: The ID of the model to use. If None, will use the first available model.
            temperature: Temperature for generation (0.0 to 1.0)
            
        Returns:
            Configured ChatOpenAI instance
        """
        if model_id is None:
            available_models = self.list_models()
            if not available_models:
                raise ValueError("No models available")
            model_id = available_models[0]

        return ChatOpenAI(
            model_name=model_id,  # Using model_name instead of model_id
            base_url=self.api_base,
            api_key=self.api_key,
            temperature=temperature)


# Example usage
if __name__ == "__main__":
    from dotenv import load_dotenv
    import os

    # # Load API key from environment
    # load_dotenv()
    # API_KEY = os.getenv("BLABLADOR_API_KEY")

    # Initialize client
    client = BlabladorClient(api_key=API_KEY)

    # List available models
    models = client.list_models()
    print(f"Available models: {models}")

    # Get a chat model
    chat_model = client.get_chat_model(model_id=models[0], temperature=0.2)

    # Make a request
    response = chat_model.invoke("What is the capital of Taiwan?")
    print(response)

{'detail': 'Not Found'}


KeyError: 'data'

In [19]:
# print(llm.invoke('Here is a fun fact about Mars:'))
response = chat_model.invoke("What is the capital of Taiwan?")
print(response.content)

AuthenticationError: Error code: 401 - {'detail': {'error': {'message': 'You must provide a valid API key. Obtain one from http://helmholtz.cloud', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}}